# 02 — Feature Engineering
Build the full feature matrix for ML models.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from src.data_loader import load_processed, train_val_test_split
from src.features import (add_calendar_features, add_fourier_features,
    add_lag_features, add_rolling_features, add_decomposition_features,
    add_temperature_features, build_feature_matrix, get_feature_columns)
plt.rcParams.update({'figure.dpi':130})
%matplotlib inline

## Load processed data

In [ ]:
df = load_processed()
print(df.shape, df.index[0], df.index[-1])

## Calendar features

In [ ]:
df_cal = add_calendar_features(df)
print(df_cal[['hour','day_of_week','month','is_weekend','is_holiday']].head(10))

## Lag features

In [ ]:
df_lag = add_lag_features(df, lags=[1,24,168])
print(df_lag[['load_mw','lag_1h','lag_24h','lag_168h']].head(10))

# Visualise lag correlation
fig, axes = plt.subplots(1,3,figsize=(14,4))
for ax, lag, label in zip(axes, ['lag_1h','lag_24h','lag_168h'],
                          ['lag 1h (t-1)','lag 24h (yesterday)','lag 168h (last week)']):
    ax.scatter(df_lag[lag], df_lag['load_mw'], s=0.5, alpha=0.2, color='#2563EB')
    ax.set_title(label); ax.set_xlabel('Lagged load (MW)'); ax.set_ylabel('Load (MW)')
plt.tight_layout()

## Fourier terms

In [ ]:
df_fourier = add_fourier_features(df)
fourier_cols = [c for c in df_fourier.columns if 'sin' in c or 'cos' in c]
print(f'{len(fourier_cols)} Fourier features created')
df_fourier[fourier_cols[:6]].plot(subplots=True, figsize=(14,8), linewidth=0.5)
plt.suptitle('Fourier features (first 6)'); plt.tight_layout()

## Build full feature matrix

In [ ]:
features = build_feature_matrix(df)
print(f'Feature matrix shape: {features.shape}')
print(f'\nAll columns:\n{list(features.columns)}')

## Feature correlation heatmap

In [ ]:
feat_cols = get_feature_columns(features)[:20]  # top 20 for readability
corr = features[feat_cols + ['load_mw']].corr()['load_mw'].drop('load_mw').sort_values()
fig, ax = plt.subplots(figsize=(8,8))
corr.plot.barh(ax=ax, color=['#DC2626' if v < 0 else '#16A34A' for v in corr])
ax.set_title('Feature correlation with load_mw')
ax.axvline(0, color='gray', lw=0.8)
plt.tight_layout()

## Train / val / test split

In [ ]:
train, val, test = train_val_test_split(features)
print(f'Train: {train.index[0]} → {train.index[-1]}  ({len(train):,} rows)')
print(f'Val:   {val.index[0]}   → {val.index[-1]}    ({len(val):,} rows)')
print(f'Test:  {test.index[0]}  → {test.index[-1]}   ({len(test):,} rows)')